In [1]:
from pybm.examples.predator_prey import  generate_synthetic_data

# Generate model and data from /home/urhp/Documents/PyBM/src/pybm/examples/predator_prey.py
model, components, times = generate_synthetic_data()
print(str(model))

# print initial values of the variables
print("Initial values of the variables:")
for var_name, var in model.vars.items():
    print(f"{var_name}: {var.initial}")  

# set initial values of the constants to good guesses
model.consts["growth_rate_prey"].initial_value = 0.5
model.consts["growth_rate_predator"].initial_value = -0.1
model.consts["predation_rate"].initial_value = 0.05
model.consts["conversion_efficiency"].initial_value = 0.05



# print initial values of the constants
print("\nInitial values of the constants:")
for const_name, const in model.consts.items():
    print(f"{const_name}: {const.initial_value}")


Model(Entities: [],
 Vars: ['n_prey', 'n_predator', 'temperature'],
 Consts: ['growth_rate_prey', 'growth_rate_predator', 'predation_rate', 'conversion_efficiency'])
Initial values of the variables:
n_prey: 40.000492061342996
n_predator: 8.016754323781468
temperature: None

Initial values of the constants:
growth_rate_prey: 0.5
growth_rate_predator: -0.1
predation_rate: 0.05
conversion_efficiency: 0.05


In [2]:
import cma
cma.CMAOptions()

{'AdaptSigma': 'True  # or False or any CMAAdaptSigmaBase class e.g. CMAAdaptSigmaTPA, CMAAdaptSigmaCSA',
 'CMA_active': 'True  # negative update, conducted after the original update',
 'CMA_active_injected': '0  #v weight multiplier for negative weights of injected solutions',
 'CMA_cmean': '1  # learning rate for the mean value',
 'CMA_const_trace': 'False  # normalize trace, 1, True, "arithm", "geom", "aeig", "geig" are valid',
 'CMA_diagonal': '0*100*N/popsize**0.5  # nb of iterations with diagonal covariance matrix, True for always',
 'CMA_diagonal_decoding': '0  # learning rate multiplier for additional diagonal update',
 'CMA_eigenmethod': 'np.linalg.eigh  # or cma.utilities.math.eig or pygsl.eigen.eigenvectors',
 'CMA_elitist': 'False  #v or "initial" or True, elitism likely impairs global search performance',
 'CMA_injections_threshold_keep_len': '1  #v keep length if Mahalanobis length is below the given relative threshold',
 'CMA_mirrors': 'popsize < 6  # values <0.5 are int

In [ ]:
from cma import fmin2, CMAEvolutionStrategy, fmin2
from pybm.estimate.int_scipy import get_data_matrix, get_initial_const_ctx, simulate
import numpy as np


vars = model.get_endo_variables()
# get the data, that we want to fit the model to
data = get_data_matrix(*vars, t_eval=times)
# initial context:
initial_ctx = get_initial_const_ctx(model)


def residuals(const_ctx):
    # get predictions
    try:
        sol = simulate(*vars, t_eval=times, const_ctx=const_ctx, max_iter=1000, method='Radau', verbose=0)
    except Exception as e:
        print(e)
        return np.array([float("inf")] * (data.size))
    pred = sol.y # of shape (n_vars, n_time_points)
    try:
        return (pred - data ).ravel()
    except Exception as e:
       # print("Error in residuals calculation")
       # print(f"pred shape: {pred.shape}, data shape: {data.shape}")
        return np.array([float("inf")] * (data.size))

evals = []

def fitness(const_ctx):
    evals.append(const_ctx)
    res = residuals(const_ctx)
    return np.sum(res**2)

bounds = (-5,5)  # bounds for the constants
# slow af
xopt, es = fmin2(fitness, initial_ctx, sigma0=0.5, options={'maxiter': 1000, 'verb_disp': 1, 
                                                            'bounds' : bounds,
                                                            'tolfun' : 1e-1})
#es = CMAEvolutionStrategy(initial_ctx, 0.5)
#es.optimize(fitness, iterations=10)

# ask and tell interface
# es = CMAEvolutionStrategy(initial_ctx, 0.5, {'bounds': bounds, 'tolfun': 1e-1, 'verb_disp': 1, 'popsize' : 5})

#for i in range(10):
#    print(f"Iteration {i+1}")
#    solutions = es.ask()
#    fitness_values = [fitness(sol) for sol in solutions]
#    es.tell(solutions, fitness_values)
#    es.disp()

(4_w,8)-aCMA-ES (mu_w=2.6,w_1=52%) in dimension 4 (seed=853858, Mon Aug  3 11:00:06 2026)
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1      8 3.534005416029526e+07 1.0e+00 4.64e-01  4e-01  5e-01 0:00.1
    2     16 2.599072234104560e+07 1.2e+00 4.65e-01  4e-01  5e-01 0:00.2


/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/scipy/integrate/_ivp/ivp.py:626: UserWarning: The following arguments have no effect for a chosen solver: `max_iter`, `verbose`.
  solver = method(fun, t0, y0, tf, vectorized=vectorized, **options)
/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/cma/utilities/utils.py:369: UserWarning: function values with index [4, 5, 6] are not finite but [np.float64(inf), np.float64(inf), np.float64(inf)]. (time=Aug  3 11:00:06 2026 class=CMAEvolutionStrategy method=ask)
  warnings.warn(msg + ' (time={}'.format(time.asctime()[4:]) +
/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/cma/utilities/utils.py:369: UserWarning: function values with index [0, 2] are not finite but [np.float64(inf), np.float64(inf)]. (time=Aug  3 11:00:06 2026 class=CMAEvolutionStrategy method=ask iteration=1)
  warnings.warn(msg + ' (time={}'.format(time.asctime()[4:]) +


    3     24 3.300055428954805e+07 1.4e+00 4.48e-01  4e-01  5e-01 0:00.2
    4     32 3.420577158663025e+07 1.5e+00 4.22e-01  3e-01  5e-01 0:00.3
    5     40 2.585650616626108e+07 1.8e+00 4.41e-01  3e-01  5e-01 0:00.4


/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/cma/utilities/utils.py:369: UserWarning: function values with index [5] are not finite but [np.float64(inf)]. (time=Aug  3 11:00:06 2026 class=CMAEvolutionStrategy method=ask iteration=2)
  warnings.warn(msg + ' (time={}'.format(time.asctime()[4:]) +
/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/cma/utilities/utils.py:369: UserWarning: function values with index [5] are not finite but [np.float64(inf)]. (time=Aug  3 11:00:06 2026 class=CMAEvolutionStrategy method=ask iteration=3)
  warnings.warn(msg + ' (time={}'.format(time.asctime()[4:]) +


    6     48 3.353353615584516e+07 1.8e+00 4.10e-01  3e-01  5e-01 0:00.5
    7     56 2.784619750293861e+07 1.8e+00 4.12e-01  3e-01  4e-01 0:00.5
    8     64 2.308192366035283e+07 1.7e+00 3.57e-01  3e-01  4e-01 0:00.6


/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/cma/utilities/utils.py:369: UserWarning: function values with index [0, 1, 4, 6, 7] are not finite but [np.float64(inf), np.float64(inf), np.float64(inf), np.float64(inf), np.float64(inf)]. (time=Aug  3 11:00:06 2026 class=CMAEvolutionStrategy method=ask iteration=5)
  warnings.warn(msg + ' (time={}'.format(time.asctime()[4:]) +
/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/cma/utilities/utils.py:369: UserWarning: function values with index [0, 5] are not finite but [np.float64(inf), np.float64(inf)]. (time=Aug  3 11:00:06 2026 class=CMAEvolutionStrategy method=ask iteration=7)
  warnings.warn(msg + ' (time={}'.format(time.asctime()[4:]) +


    9     72 3.022624425981091e+07 1.7e+00 3.54e-01  2e-01  4e-01 0:00.7
   10     80 3.282236979720378e+07 2.1e+00 3.21e-01  2e-01  4e-01 0:00.8
termination on {'maxiter': 10} (Mon Aug  3 11:00:07 2026)
final/bestever f-value = 3.404626e+07 2.308192e+07 after 81/63 evaluations
incumbent solution: [-0.20221863, -0.23049597, -0.8445298, 0.37853631]
std deviations: [0.19828838, 0.23853269, 0.36417827, 0.22307579]


/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/cma/utilities/utils.py:369: UserWarning: function values with index [2] are not finite but [np.float64(inf)]. (time=Aug  3 11:00:07 2026 class=CMAEvolutionStrategy method=ask iteration=9)
  warnings.warn(msg + ' (time={}'.format(time.asctime()[4:]) +
